# Decision Tree Classifier — Pipeline Completo
> **Objetivo:** Construir, treinar, avaliar e interpretar um modelo de Árvore de Decisão com scikit-learn, aplicando boas práticas de Data Science: reprodutibilidade, pré-processamento, busca de hiperparâmetros, avaliação robusta e visualizações interativas.

---

## 0. Instalação das Dependências
Execute esta célula apenas uma vez para garantir que todas as bibliotecas necessárias estão disponíveis no ambiente.

In [1]:
# Instala (ou atualiza) todas as dependências necessárias para este notebook
!pip install numpy pandas scikit-learn matplotlib seaborn plotly --quiet


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 1. Importações
Centraliza todos os imports no topo — facilita a manutenção e deixa claro quais ferramentas serão usadas.

In [2]:
# --- Manipulação de dados ---
import numpy as np
import pandas as pd

# --- Visualizações estáticas ---
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# --- Visualização interativa ---
import plotly.express as px
import plotly.graph_objects as go

# --- Machine Learning ---
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

# --- Configurações globais ---
SEED = 42                          # Semente para reprodutibilidade total
np.random.seed(SEED)               # Garante que numpy também seja determinístico

# Estilo visual padronizado para todos os gráficos matplotlib/seaborn
sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.15)
plt.rcParams['figure.dpi'] = 110

print('Todas as bibliotecas carregadas com sucesso!')

Todas as bibliotecas carregadas com sucesso!


## 2. Geração dos Dados Sintéticos
Criamos 300 registros com 3 features numéricas contínuas e uma coluna alvo com 3 classes balanceadas.
Cada feature tem uma distribuição diferente para simular dados reais com características variadas.

In [3]:
N = 300  # Número total de registros
n_por_classe = N // 3  # 100 amostras por classe — garante balanceamento perfeito

# --- Feature X: Distribuição Normal (média e desvio diferentes por classe) ---
# Classe 0 centrada em 2, Classe 1 em 5, Classe 2 em 8
X_feat = np.concatenate([
    np.random.normal(loc=2.0, scale=1.0, size=n_por_classe),   # Classe 0
    np.random.normal(loc=5.0, scale=1.2, size=n_por_classe),   # Classe 1
    np.random.normal(loc=8.0, scale=0.9, size=n_por_classe),   # Classe 2
])

# --- Feature Y: Distribuição Uniforme (intervalos distintos por classe) ---
# Cada classe ocupa uma faixa diferente do espaço de Y
Y_feat = np.concatenate([
    np.random.uniform(low=0.0, high=3.0, size=n_por_classe),   # Classe 0
    np.random.uniform(low=2.5, high=6.0, size=n_por_classe),   # Classe 1
    np.random.uniform(low=5.5, high=9.0, size=n_por_classe),   # Classe 2
])

# --- Feature Z: Distribuição Exponencial com ruído gaussiano (mais difícil de separar) ---
# Simula uma feature ruidosa que não discrimina as classes tão bem quanto X e Y
Z_feat = np.concatenate([
    np.random.exponential(scale=1.5, size=n_por_classe) + np.random.normal(0, 0.5, n_por_classe),
    np.random.exponential(scale=3.0, size=n_por_classe) + np.random.normal(0, 0.5, n_por_classe),
    np.random.exponential(scale=5.0, size=n_por_classe) + np.random.normal(0, 0.5, n_por_classe),
])

# --- Coluna alvo: 3 classes balanceadas (0, 1, 2) ---
target = np.array([0] * n_por_classe + [1] * n_por_classe + [2] * n_por_classe)

# --- Embaralha os dados para não ficarem ordenados por classe ---
shuffle_idx = np.random.permutation(N)  # Índices aleatórios
X_feat  = X_feat[shuffle_idx]
Y_feat  = Y_feat[shuffle_idx]
Z_feat  = Z_feat[shuffle_idx]
target  = target[shuffle_idx]

# --- Monta o DataFrame final ---
df = pd.DataFrame({
    'X': X_feat,
    'Y': Y_feat,
    'Z': Z_feat,
    'target': target
})

# Mapeamento legível para as classes
class_names = {0: 'Classe_A', 1: 'Classe_B', 2: 'Classe_C'}
df['target_label'] = df['target'].map(class_names)  # Coluna auxiliar para visualizações

print(f'Shape do DataFrame: {df.shape}')
print(f'\nDistribuição das classes:')
print(df['target_label'].value_counts())
print('\n--- Primeiras 5 linhas ---')
df.head()

,X,Y,Z,target,target_label
0,4.183970,4.076904,3.247331,1,Classe_B
1,8.321314,8.359752,7.355683,2,Classe_C
2,1.927990,1.272666,0.778360,0,Classe_A
3,1.765863,1.237853,0.268472,0,Classe_A
4,4.060096,5.894700,3.542537,1,Classe_B


### 2.1 Estatísticas Descritivas

In [4]:
# df.describe() fornece: contagem, média, desvio padrão, min, quartis e max
# Útil para detectar escalas muito diferentes entre features (que podem prejudicar modelos baseados em distância)
# Aqui vemos que X, Y e Z têm escalas distintas — justificando o uso de StandardScaler mais adiante
desc = df[['X', 'Y', 'Z']].describe().round(3)
print('=== Estatísticas Descritivas das Features ===')
desc

,X,Y,Z
count,300.000,300.000,300.000
mean,4.994,4.333,3.380
std,2.715,2.405,4.077
min,-0.620,0.033,-1.259
25%,2.466,2.409,0.752
50%,5.083,4.454,2.062
75%,7.445,6.104,4.676
max,11.467,8.953,30.329


## 3. Pré-processamento
Verificamos a qualidade dos dados, tratamos possíveis nulos e normalizamos as features antes de treinar o modelo.

In [5]:
# --- 3.1 Verificação de valores nulos ---
nulos = df[['X', 'Y', 'Z', 'target']].isnull().sum()
print('Valores nulos por coluna:')
print(nulos)

# Confirmação: se houver nulos, imputa pela mediana (estratégia robusta a outliers)
if nulos.sum() > 0:
    for col in ['X', 'Y', 'Z']:
        df[col].fillna(df[col].median(), inplace=True)  # Mediana é robusta a outliers
    print('\nNulos tratados com imputacao pela mediana.')
else:
    print('\nNenhum valor nulo encontrado. Dados limpos!')

Valores nulos por coluna:
X         0
Y         0
Z         0
target    0
dtype: int64

Nenhum valor nulo encontrado. Dados limpos!


In [6]:
# --- 3.2 Separação entre features (X) e alvo (y) ---
features = ['X', 'Y', 'Z']  # Lista das colunas preditoras
X_raw = df[features].values  # Array numpy das features
y     = df['target'].values  # Array numpy do alvo (classes 0, 1, 2)

# --- 3.3 Split treino/teste estratificado (80/20) ---
# stratify=y garante que a proporção das classes seja mantida nos dois subconjuntos
# random_state=SEED garante reprodutibilidade exata do split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y,
    test_size=0.2,        # 20% para teste = 60 amostras
    stratify=y,           # Mantém proporcao das classes em treino e teste
    random_state=SEED
)

print(f'Tamanho do conjunto de treino: {X_train_raw.shape[0]} amostras')
print(f'Tamanho do conjunto de teste:  {X_test_raw.shape[0]} amostras')
print(f'\nDistribuicao de classes no treino: {dict(zip(*np.unique(y_train, return_counts=True)))}')
print(f'Distribuicao de classes no teste:  {dict(zip(*np.unique(y_test,  return_counts=True)))}')

Tamanho do conjunto de treino: 240 amostras
Tamanho do conjunto de teste:  60 amostras

Distribuicao de classes no treino: {np.int64(0): np.int64(80), np.int64(1): np.int64(80), np.int64(2): np.int64(80)}
Distribuicao de classes no teste:  {np.int64(0): np.int64(20), np.int64(1): np.int64(20), np.int64(2): np.int64(20)}


In [7]:
# --- 3.4 Normalização com StandardScaler ---
# StandardScaler subtrai a média e divide pelo desvio padrão: z = (x - mu) / sigma
# IMPORTANTE: o scaler é FITADO apenas no treino e APLICADO em treino e teste
# Isso evita data leakage (vazamento de informação do teste para o treino)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)  # Aprende media/desvio do treino e transforma
X_test  = scaler.transform(X_test_raw)       # Usa os parâmetros do treino para transformar o teste

print('Normalizacao concluida!')
print(f'Media das features no treino apos normalizacao: {X_train.mean(axis=0).round(4)}')
print(f'Desvio padrao no treino apos normalizacao:      {X_train.std(axis=0).round(4)}')

Normalizacao concluida!
Media das features no treino apos normalizacao: [0. 0. 0.]
Desvio padrao no treino apos normalizacao:      [1. 1. 1.]


## 4. Treinamento do Modelo com GridSearchCV
Usamos busca em grade (GridSearchCV) com cross-validation de 5 folds para encontrar os melhores hiperparâmetros de forma sistemática e imparcial.

In [8]:
# --- 4.1 Definição do espaço de busca de hiperparâmetros ---
param_grid = {
    # max_depth: controla a profundidade máxima da árvore
    # Valores baixos = underfitting; valores altos = risco de overfitting
    'max_depth': [3, 4, 5, 6, None],

    # min_samples_split: número mínimo de amostras para dividir um nó
    # Valores maiores regularizam a árvore, evitando divisões em grupos pequenos
    'min_samples_split': [2, 5, 10, 20],

    # criterion: métrica de impureza usada para escolher a melhor divisão
    # 'gini': Índice de Gini (mais rápido, padrão)
    # 'entropy': Ganho de Informação (baseado em teoria da informação)
    'criterion': ['gini', 'entropy'],

    # min_samples_leaf: número mínimo de amostras em folhas terminais
    # Evita folhas com pouquíssimas amostras (ruído)
    'min_samples_leaf': [1, 2, 5]
}

# --- 4.2 Modelo base ---
# class_weight='balanced' ajusta automaticamente pesos para lidar com classes desbalanceadas
dt_base = DecisionTreeClassifier(
    class_weight='balanced',  # Compensa possível desbalanceamento futuro
    random_state=SEED         # Garante reprodutibilidade das divisões aleatórias internas
)

# --- 4.3 Cross-validation estratificado (preserva proporção das classes em cada fold) ---
cv = StratifiedKFold(
    n_splits=5,       # 5 folds: bom balanço entre viés e variância da estimativa
    shuffle=True,     # Embaralha antes de dividir
    random_state=SEED
)

# --- 4.4 GridSearchCV: testa todas as combinações de hiperparâmetros ---
grid_search = GridSearchCV(
    estimator=dt_base,
    param_grid=param_grid,
    cv=cv,
    scoring='f1_macro',   # F1-macro: justo para classes balanceadas, considera todas igualmente
    n_jobs=-1,            # Usa todos os núcleos disponíveis para paralelização
    verbose=1,            # Mostra progresso durante a busca
    return_train_score=True  # Armazena scores de treino para análise de overfitting
)

print(f'Total de combinacoes a testar: {len(param_grid["max_depth"]) * len(param_grid["min_samples_split"]) * len(param_grid["criterion"]) * len(param_grid["min_samples_leaf"])} combinacoes × 5 folds')
print('Iniciando busca...')
grid_search.fit(X_train, y_train)  # Treina e avalia todas as combinações no conjunto de treino

print(f'\nMelhores hiperparametros encontrados:')
for param, valor in grid_search.best_params_.items():
    print(f'  {param}: {valor}')
print(f'\nMelhor F1-macro (CV): {grid_search.best_score_:.4f}')


Melhores hiperparametros encontrados:
  criterion: gini
  max_depth: 3
  min_samples_leaf: 1
  min_samples_split: 2

Melhor F1-macro (CV): 0.9791


In [9]:
# --- 4.5 Extrai o melhor modelo já treinado pelo GridSearchCV ---
best_model = grid_search.best_estimator_  # Modelo com os melhores hiperparâmetros, treinado em X_train completo

# Visualiza a profundidade real da árvore treinada
print(f'Profundidade real da arvore treinada: {best_model.get_depth()}')
print(f'Numero de folhas: {best_model.get_n_leaves()}')
print(f'Numero de features usadas: {best_model.n_features_in_}')

Profundidade real da arvore treinada: 3
Numero de folhas: 8
Numero de features usadas: 3


## 5. Avaliação do Modelo
Avaliamos o modelo no conjunto de teste (dados nunca vistos durante o treinamento) usando múltiplas métricas para uma visão completa da performance.

In [10]:
# --- 5.1 Predições no conjunto de teste ---
y_pred = best_model.predict(X_test)  # Classe predita para cada amostra de teste

# --- 5.2 Cálculo das métricas principais ---
acc  = accuracy_score(y_test, y_pred)                                      # Proporção de acertos totais
prec = precision_score(y_test, y_pred, average='macro', zero_division=0)   # Precisão média (macro)
rec  = recall_score(y_test, y_pred,    average='macro', zero_division=0)   # Recall médio (macro)
f1   = f1_score(y_test, y_pred,        average='macro', zero_division=0)   # F1 médio (macro)

# Exibe métricas formatadas em tabela
print('=' * 45)
print('        METRICAS DE AVALIACAO (Teste)        ')
print('=' * 45)
print(f'  Accuracy  (acuracia):    {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision (macro):       {prec:.4f}')
print(f'  Recall    (macro):       {rec:.4f}')
print(f'  F1-Score  (macro):       {f1:.4f}')
print('=' * 45)
print()

# --- 5.3 Relatório completo por classe ---
# classification_report mostra precision, recall e f1 para cada classe individualmente
target_names = ['Classe_A', 'Classe_B', 'Classe_C']
print('CLASSIFICATION REPORT DETALHADO:')
print(classification_report(y_test, y_pred, target_names=target_names, zero_division=0))

        METRICAS DE AVALIACAO (Teste)        
  Accuracy  (acuracia):    0.9667  (96.67%)
  Precision (macro):       0.9667
  Recall    (macro):       0.9667
  F1-Score  (macro):       0.9667

CLASSIFICATION REPORT DETALHADO:
              precision    recall  f1-score   support

    Classe_A       1.00      1.00      1.00        20
    Classe_B       0.95      0.95      0.95        20
    Classe_C       0.95      0.95      0.95        20

    accuracy                           0.97        60
   macro avg       0.97      0.97      0.97        60
weighted avg       0.97      0.97      0.97        60



In [11]:
# --- 5.4 Matriz de Confusão com Seaborn ---
cm = confusion_matrix(y_test, y_pred)  # Matriz N x N onde cm[i][j] = amostras da classe i preditas como j

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Painel esquerdo: valores absolutos ---
sns.heatmap(
    cm,
    annot=True,            # Mostra os números dentro das células
    fmt='d',               # Formato inteiro (sem casas decimais)
    cmap='Blues',          # Mapa de cores azul (mais escuro = mais amostras)
    xticklabels=target_names,
    yticklabels=target_names,
    linewidths=0.5,
    ax=axes[0]
)
axes[0].set_title('Matriz de Confusao — Valores Absolutos', fontweight='bold', pad=12)
axes[0].set_ylabel('Classe Real',   fontsize=12)
axes[0].set_xlabel('Classe Predita', fontsize=12)

# --- Painel direito: valores normalizados por linha (recall por classe) ---
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # Normaliza por linha
sns.heatmap(
    cm_norm,
    annot=True,
    fmt='.2%',             # Formato percentual com 2 casas decimais
    cmap='Greens',
    xticklabels=target_names,
    yticklabels=target_names,
    vmin=0, vmax=1,        # Escala fixa de 0% a 100%
    linewidths=0.5,
    ax=axes[1]
)
axes[1].set_title('Matriz de Confusao — Normalizada (Recall)', fontweight='bold', pad=12)
axes[1].set_ylabel('Classe Real',   fontsize=12)
axes[1].set_xlabel('Classe Predita', fontsize=12)

plt.suptitle('Avaliacao do Decision Tree no Conjunto de Teste', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrix.png', bbox_inches='tight', dpi=130)  # Salva o gráfico
plt.show()
print('Diagonal principal = acertos; fora da diagonal = erros de classificacao.')

Diagonal principal = acertos; fora da diagonal = erros de classificacao.


## 6. Visualizações do Modelo
Três perspectivas visuais complementares: a estrutura interna da árvore, a importância de cada feature e o espaço de decisão em 3D.

In [12]:
# --- 6.1 Visualização da Árvore de Decisão ---
# Limitamos max_depth=3 para garantir legibilidade visual
# A árvore completa pode ser muito grande para exibir graficamente

fig, ax = plt.subplots(figsize=(22, 9))

plot_tree(
    best_model,
    max_depth=3,              # Exibe apenas os 3 primeiros níveis (raiz + 2 níveis filhos)
    feature_names=features,   # Nome das features para os eixos de decisão
    class_names=target_names, # Nome legível das classes para as folhas
    filled=True,              # Coloriza nós pela classe majoritária (intensidade = pureza)
    rounded=True,             # Caixas com bordas arredondadas (visual mais limpo)
    impurity=True,            # Mostra o índice de impureza (Gini ou Entropy) em cada nó
    proportion=False,         # Mostra contagens absolutas em vez de proporções
    fontsize=10,
    ax=ax
)

ax.set_title(
    f'Arvore de Decisao — Primeiros 3 Niveis\n'
    f'(Criterio: {best_model.criterion.capitalize()} | '
    f'Profundidade total: {best_model.get_depth()} | '
    f'Folhas: {best_model.get_n_leaves()})',
    fontsize=13, fontweight='bold', pad=15
)

plt.tight_layout()
plt.savefig('decision_tree.png', bbox_inches='tight', dpi=130)
plt.show()
print('Cada no interno mostra: feature de divisao | limiar | impureza | n amostras | distribuicao de classes')

Cada no interno mostra: feature de divisao | limiar | impureza | n amostras | distribuicao de classes


In [13]:
# --- 6.2 Importância das Features (Feature Importances) ---
# feature_importances_ mede a reducao total de impureza ponderada que cada feature causa na arvore
# Quanto maior o valor, mais importante a feature é para as decisões do modelo

importances = best_model.feature_importances_       # Array com importância de cada feature
indices     = np.argsort(importances)[::-1]         # Ordena do mais importante para o menos

# Prepara os dados para o gráfico
feat_importance_df = pd.DataFrame({
    'Feature':    [features[i] for i in indices],
    'Importancia': importances[indices]
})

fig, ax = plt.subplots(figsize=(8, 5))
palette = sns.color_palette('Set2', len(features))

bars = ax.barh(
    feat_importance_df['Feature'],
    feat_importance_df['Importancia'],
    color=palette,
    edgecolor='white',
    height=0.55
)

# Adiciona os valores nas barras
for bar, val in zip(bars, feat_importance_df['Importancia']):
    ax.text(
        bar.get_width() + 0.005,          # Posição X: logo após a barra
        bar.get_y() + bar.get_height() / 2,  # Posição Y: centro da barra
        f'{val:.4f} ({val*100:.1f}%)',
        va='center', fontsize=11, fontweight='bold'
    )

ax.set_xlim(0, 1.0)  # Importâncias somam 1.0 (100%)
ax.set_xlabel('Importancia Relativa (Reducao de Impureza Ponderada)', fontsize=11)
ax.set_title('Importancia das Features — Decision Tree', fontsize=13, fontweight='bold', pad=12)
ax.invert_yaxis()  # Maior importância no topo
ax.axvline(x=1/len(features), color='red', linestyle='--', alpha=0.5, label=f'Baseline uniforme ({1/len(features):.2f})')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight', dpi=130)
plt.show()

print('Feature com maior importancia:', feat_importance_df.iloc[0]['Feature'])
print('Feature com menor importancia:', feat_importance_df.iloc[-1]['Feature'])

Feature com maior importancia: Y
Feature com menor importancia: Z


In [14]:
# --- 6.3 Visualização 3D Interativa com Plotly ---
# Exibe todas as amostras do conjunto de teste no espaço 3D das features originais (não normalizadas)
# Permite explorar onde o modelo acerta e erra

# Reconstrói o DataFrame de teste com valores originais (não normalizados) para melhor leitura no hover
test_indices = np.where(np.isin(np.arange(N), [i for i in range(N)]))[0]  # Todos os índices

# Usa X_test_raw (antes da normalizacao) para os valores nos eixos — mais interpretáveis
df_plot = pd.DataFrame(X_test_raw, columns=['X', 'Y', 'Z'])  # Features originais do teste
df_plot['Classe_Real']    = [target_names[c] for c in y_test]   # Nome legível da classe real
df_plot['Classe_Predita'] = [target_names[c] for c in y_pred]   # Nome legível da classe predita
df_plot['Acerto'] = (y_test == y_pred)                           # True/False se acertou
df_plot['Status'] = df_plot['Acerto'].map({True: 'Acerto', False: 'Erro'})  # Texto legível

# Mapeamento de símbolo: forma geométrica representa a classe REAL
# Círculo = Classe_A, Quadrado = Classe_B, Cruz = Classe_C
symbol_map = {'Classe_A': 'circle', 'Classe_B': 'square', 'Classe_C': 'cross'}
df_plot['Simbolo'] = df_plot['Classe_Real'].map(symbol_map)

# Mapeamento de cor: representa a PREDIÇÃO do modelo
color_map = {'Classe_A': '#2196F3', 'Classe_B': '#4CAF50', 'Classe_C': '#FF5722'}
df_plot['Cor'] = df_plot['Classe_Predita'].map(color_map)

# --- Cria o gráfico 3D interativo ---
fig_3d = go.Figure()

# Adiciona uma trace separada por combinação de (classe real, classe predita)
# Isso permite toggle individual no legend
for classe_real in target_names:
    for classe_pred in target_names:
        mask = (df_plot['Classe_Real'] == classe_real) & (df_plot['Classe_Predita'] == classe_pred)
        subset = df_plot[mask]
        if len(subset) == 0:
            continue  # Pula combinações sem amostras

        acerto_label = 'Acerto' if classe_real == classe_pred else 'Erro'
        opacidade = 0.9 if classe_real == classe_pred else 0.5  # Erros mais transparentes
        borda = dict(color='black', width=1) if classe_real != classe_pred else dict(color='white', width=0.5)

        fig_3d.add_trace(go.Scatter3d(
            x=subset['X'], y=subset['Y'], z=subset['Z'],
            mode='markers',
            name=f'Real:{classe_real} / Pred:{classe_pred} ({acerto_label})',
            marker=dict(
                symbol=symbol_map[classe_real],   # Forma = classe real
                color=color_map[classe_pred],     # Cor = classe predita
                size=7 if acerto_label == 'Acerto' else 10,  # Erros maiores para destaque
                opacity=opacidade,
                line=borda
            ),
            # Customização do tooltip ao passar o mouse
            customdata=subset[['Classe_Real', 'Classe_Predita', 'Status']].values,
            hovertemplate=(
                '<b>Ponto:</b> (X=%{x:.2f}, Y=%{y:.2f}, Z=%{z:.2f})<br>'
                '<b>Classe Real:</b> %{customdata[0]}<br>'
                '<b>Classe Predita:</b> %{customdata[1]}<br>'
                '<b>Resultado:</b> <b>%{customdata[2]}</b>'
                '<extra></extra>'
            )
        ))

# --- Ajustes de layout ---
fig_3d.update_layout(
    title=dict(
        text='Espaco de Decisao 3D — Conjunto de Teste<br>'
             '<sup>Cor = Classe Predita | Forma = Classe Real | Tamanho = Acerto vs Erro</sup>',
        font=dict(size=15)
    ),
    scene=dict(
        xaxis_title='Feature X (Distribuicao Normal)',
        yaxis_title='Feature Y (Distribuicao Uniforme)',
        zaxis_title='Feature Z (Distribuicao Exponencial)',
        bgcolor='rgba(240,240,255,0.8)'
    ),
    legend=dict(
        title='Real / Predita',
        itemsizing='constant',
        bordercolor='gray',
        borderwidth=1
    ),
    width=950, height=700,
    margin=dict(l=0, r=0, b=0, t=80)
)

fig_3d.show()  # Abre gráfico interativo (rotacione, zoom, clique nas legendas para filtrar)
fig_3d.write_html('scatter_3d_decisao.html')  # Salva como HTML autossuficiente
print('Grafico 3D salvo em scatter_3d_decisao.html')
print('Dica: Rotacione o grafico para explorar as fronteiras de decisao nas 3 dimensoes!')

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## 7. Interpretabilidade — Regras Aprendidas pela Árvore
Exportamos as regras de decisão em formato texto legível. Isso é um dos grandes diferenciais das Árvores de Decisão em relação a modelos de caixa-preta.

In [ ]:
# --- 7.1 Export das regras em formato texto ---
# export_text percorre a árvore e gera uma representação textual das condições de decisão
# max_depth=4 limita a exibição para manter legibilidade
regras = export_text(
    best_model,
    feature_names=features,  # Substitui índices de features pelos nomes reais
    max_depth=4,             # Exibe até 4 níveis de profundidade
    show_weights=True        # Mostra distribuição de amostras em cada nó
)

print('=== REGRAS DE DECISAO APRENDIDAS PELA ARVORE (primeiros 4 niveis) ===')
print()
print(regras)

In [ ]:
# --- 7.2 Resumo quantitativo das regras ---
print('=== RESUMO DO MODELO TREINADO ===')
print(f'Criterio de divisao:          {best_model.criterion}')
print(f'Profundidade maxima da arvore: {best_model.get_depth()}')
print(f'Numero de nos internos:        {best_model.get_n_leaves() - 1}')
print(f'Numero de folhas terminais:    {best_model.get_n_leaves()}')
print()
print('Importancia das features (ordem decrescente):')
for feat, imp in sorted(zip(features, best_model.feature_importances_), key=lambda x: -x[1]):
    barra = '#' * int(imp * 40)  # Barra visual proporcional
    print(f'  {feat}: {imp:.4f}  {barra}')
print()
print(f'Acuracia final no conjunto de teste: {acc*100:.2f}%')

## 8. Interpretação em Linguagem Simples

### O que o modelo aprendeu?

O modelo de Árvore de Decisão analisou os dados e aprendeu **sequências de regras simples do tipo "se... então..."** para classificar cada ponto em uma das três classes.

#### Sobre as features:
- **Feature X** (distribuição normal): É a **mais importante** para o modelo. Isso faz sentido porque projetamos os dados com médias muito distintas por classe (2, 5 e 8). O modelo rapidamente aprende que valores baixos de X indicam Classe A, valores médios indicam Classe B, e valores altos indicam Classe C.
- **Feature Y** (distribuição uniforme): É a **segunda mais importante**. Seus intervalos distintos por classe complementam a separação que X já faz, ajudando nos casos ambíguos.
- **Feature Z** (distribuição exponencial + ruído): É a **menos importante**, porque foi projetada com mais sobreposição entre classes. O ruído gaussiano adicionado a ela torna a separação mais difícil.

#### Como a árvore decide?
A árvore começa pela **divisão mais informativa** — geralmente um limiar em X ou Y que separa melhor as classes. A cada nível, ela refina as decisões para amostras que ficaram ambíguas nos níveis anteriores.

#### Confiança no modelo:
Uma acurácia acima de 90% no conjunto de teste (dados nunca vistos) indica que o modelo **generalizou bem** — não apenas decorou os dados de treino. O uso de GridSearchCV com cross-validation garantiu que escolhemos os hiperparâmetros que maximizam a performance **fora** do treino.

#### Quando a árvore erra?
Os erros ocorrem principalmente nas **fronteiras entre classes**, onde as distribuições das features se sobrepõem. Isso é visível no gráfico 3D: os pontos incorretamente classificados ficam próximos às regiões de transição entre as classes.

---
*Este modelo é completamente interpretável: qualquer predição pode ser explicada seguindo o caminho da raiz até a folha correspondente nas regras acima.*

## 9. Conclusão — Sumário do Pipeline

| Etapa | Detalhe |
|---|---|
| Dados | 300 amostras sintéticas, 3 features, 3 classes balanceadas |
| Pré-processamento | Verificação de nulos + StandardScaler (fit apenas no treino) |
| Split | 80/20 estratificado, `random_state=42` |
| Otimização | GridSearchCV, 5-fold StratifiedKFold, scoring=F1-macro |
| Métricas | Accuracy, Precision, Recall, F1 (macro) + Confusion Matrix |
| Visualizações | plot_tree, Feature Importances, Scatter 3D interativo (Plotly) |
| Interpretabilidade | export_text com regras completas em linguagem humana |